# Round-2 revision analyses (Colab)

Run the cells **from top to bottom**. Every section saves its tables to `rugpull_research/rev2/rev2_colab_results.xlsx`
as soon as it finishes, so nothing is lost if Colab disconnects. Section 2 (Slither) can be re-run and
continues where it stopped.

| Section | What it answers | Time |
|---|---|---|
| 0 | Setup | ~3 min |
| 1 | Reproduce the published model (MCC 0.840) | ~2 min |
| 2 | Re-run Slither and **store** the detector output; parity check | ~1–2 h |
| 3 | Detector-level breakdown; version-dependent detectors (Reviewer A) | ~5 min |
| 4 | Enriched metadata baseline (Reviewer B) | ~3 min |
| 5 | Source-code duplicates and similarity; grouped CV; temporal split (Reviewer B) | ~10 min |
| 6 | Regex variants: comments stripped, contract-only code (Reviewers A, B) | ~5 min |
| 7 | Files for the manual work: regex annotation, top-20 legit audit, missed rug pulls | ~2 min |
| 8 | Legitimate tokens with 180–365 days of activity (needs Etherscan key) | ~5 min |
| 9 | **Run later**, after the annotation sheet is filled in | 1 min |

## 0. Setup

In [ ]:
DRY_RUN = False          # leave False in Colab

import os, sys, json, time, shutil
if not DRY_RUN:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'slither-analyzer==0.11.6', 'solc-select==1.2.0',
                    'xgboost==3.4.1', 'scikit-learn==1.6.1', 'shap==0.54.1', 'openpyxl'], check=True)
    BASE  = '/content/drive/MyDrive/rugpull_research'
    LOCAL = '/content/r2work'
else:
    BASE  = os.environ['R2_BASE']; LOCAL = os.environ['R2_LOCAL']

OUT = f'{BASE}/rev2'
for p in [OUT, f'{OUT}/slither_json', f'{OUT}/annotation/contracts', LOCAL, f'{LOCAL}/src']:
    os.makedirs(p, exist_ok=True)
sys.path.insert(0, OUT)
assert os.path.exists(f'{OUT}/rev2lib.py'), 'Upload rev2lib.py into rugpull_research/rev2/ first'
from rev2lib import *
import numpy as np, pandas as pd

RAW    = f'{BASE}/raw_etherscan'
PATHS  = dict(D=f'{BASE}/features/D_single_era_matched.csv',
              sun=f'{BASE}/dataset/dianxiang/rugpull_full_dataset_new.csv',
              crp=f'{BASE}/dataset/crpwarner/groundtruth/groundTruth.xlsx',
              cand_span=f'{BASE}/features/cand_span.csv',
              legit886=f'{BASE}/repo/data/legit_candidates_processed_886.csv')
for k, p in PATHS.items(): print(f'{k:10s}', 'OK ' if os.path.exists(p) else 'MISSING', p)
print('raw_etherscan files:', len(os.listdir(RAW)))

RESULTS = {}
XLSX = f'{OUT}/rev2_colab_results.xlsx'
def save(name, df, index=False):
    """Store a table and rewrite the results workbook."""
    RESULTS[name] = (df, index)
    with pd.ExcelWriter(XLSX) as w:
        for k, (v, ix) in RESULTS.items(): v.to_excel(w, sheet_name=k[:31], index=ix)
    print(f'saved sheet "{name}"')

## 1. Data and the published model

In [ ]:
D = pd.read_csv(PATHS['D']); D['address'] = D['address'].str.lower()
assert len(D) == 2278 and D.label.sum() == 1864, D.label.value_counts()
sun = pd.read_csv(PATHS['sun']); sun['address'] = sun['address'].str.lower()
TRAP = 'From Programming Bugs to Multimillion-Dollar Scams: An Analysis of Trapdoor Tokens on Uniswap'
D = D.merge(sun[['address', 'Type', 'Sources']].drop_duplicates('address'), on='address', how='left')
D['group'] = np.where(D.label == 0, 'Legitimate',
             np.where(D.Type == 'Transaction Limitation', 'Rug: Transaction Limitation',
             np.where(D.Type == 'Fee', 'Rug: Fee', 'Rug: other or CRPWarner only')))
print(D.group.value_counts())

y = D.label.values; G0 = feature_groups(D)
print('feature-space clusters:', len(set(G0)))
F0, O0 = cv_folds(D[FEAT], y, G0)
print('published model:', summarise(F0))               # expected MCC 0.840 ± 0.043
assert abs(F0.mcc.mean() - 0.840) < 0.002, 'Model does not reproduce the paper; stop and report'
D['p_oof'] = O0.groupby('i').p.mean().sort_index().values
D['pred_oof'] = (D.p_oof >= 0.5).astype(int)
save('1_reproduction', pd.DataFrame([{'set': 'published model'} | summarise(F0)]))

## 2. Re-run Slither and store the full detector output
Identical sources compiled with the same compiler are analysed once and the result is reused.
The cell can be re-run after a disconnect; finished contracts are skipped.

In [ ]:
import concurrent.futures as cf
info = []
for a in D.address:
    res, px = eff_res(RAW, a)
    files, remaps = unpack(res)
    info.append(dict(address=a, res=res, is_proxy_chk=px, files=files, remaps=remaps, ver=solc_ver(res),
                     main=pick_main(files, res.get('ContractName')), name=res.get('ContractName')))
INFO = {r['address']: r for r in info}
assert (pd.Series({a: r['is_proxy_chk'] for a, r in INFO.items()}).reindex(D.address).values == D.is_proxy.values).all()

if not DRY_RUN:
    for v in sorted({r['ver'] for r in info}):
        subprocess.run(['solc-select', 'install', v], capture_output=True)
    SOLC_PATH = lambda v: None
else:
    SOLC_PATH = lambda v: f"{os.environ['R2_SOLC']}/solc-{v}"

def job_key(r):
    return sha(json.dumps([sorted(r['files'].items()), r['main'], r['ver'], r['remaps']]))

def run_one(a):
    out = f'{OUT}/slither_json/{a}.json.gz'
    if os.path.exists(out): return a, 'cached'
    r = INFO[a]; k = job_key(r); kp = f'{LOCAL}/bykey_{k}.json.gz'
    if os.path.exists(kp):
        shutil.copy(kp, out); return a, 'reused'
    d = write_tree(f'{LOCAL}/src', a, r['files'])
    det, st = run_slither_json(d, r['main'], r['ver'], r['remaps'], solc_path=SOLC_PATH(r['ver']))
    obj = dict(status=st, detectors=det, ver=r['ver'], main=r['main'])
    save_json_gz(obj, kp); shutil.copy(kp, out)
    return a, st

# run identical sources once: first one address per key, then the rest
order = pd.Series({a: job_key(r) for a, r in INFO.items()})
first = order.drop_duplicates().index.tolist(); rest = [a for a in order.index if a not in set(first)]
print('contracts:', len(order), '| distinct source/compiler combinations:', len(first))
t0 = time.time(); stat = []
for batch in (first, rest):
    with cf.ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
        for n, (a, st) in enumerate(ex.map(run_one, batch), 1):
            stat.append((a, st))
            if n % 200 == 0: print(n, '/', len(batch), f'{(time.time()-t0)/60:.1f} min')
print(pd.Series([s for _, s in stat]).value_counts())

In [ ]:
# Parity: the stored output must give exactly the published features
DET = {}; bad = []
for a in D.address:
    obj = load_json_gz(f'{OUT}/slither_json/{a}.json.gz'); DET[a] = obj['detectors']
    if obj['status'] != 'ok': bad.append((a, 'status', obj['status'])); continue
    f = slither_feats(obj['detectors']); f.update(text_feats('\n'.join(INFO[a]['files'].values())))
    row = D.loc[D.address == a].iloc[0]
    bad += [(a, k, (f[k], row[k])) for k in FEAT if float(f[k]) != float(row[k])]
print('mismatches:', len(bad))
PARITY_OK = len(bad) == 0
save('2_parity', pd.DataFrame(bad, columns=['address', 'feature', 'new_vs_published']) if bad else
     pd.DataFrame([{'result': 'all 2,278 contracts x 29 features identical'}]))
if not PARITY_OK: print('PARITY FAILED: send the sheet "2_parity" before going further'); print(bad[:20])

## 3. Which Slither detectors make up the counts, and does it depend on the compiler?

In [ ]:
long = pd.DataFrame([(a, x['check'], x['impact']) for a in D.address for x in DET[a]], columns=['address', 'check', 'impact'])
long = long.merge(D[['address', 'label', 'solc_minor']], on='address')
cnt = long.groupby(['address', 'check']).size().unstack(fill_value=0).reindex(D.address, fill_value=0)
imp_of = long.drop_duplicates('check').set_index('check').impact

rows = []
for chk in cnt.columns:
    c = cnt[chk]; r = c[D.label.values == 1]; l = c[D.label.values == 0]
    rows.append(dict(check=chk, impact=imp_of[chk], rug_prev=(r > 0).mean(), legit_prev=(l > 0).mean(),
                     rug_mean=r.mean(), legit_mean=l.mean(),
                     **{f'legit_prev_0.{v}': (c[(D.label.values == 0) & (D.solc_minor.values == v)] > 0).mean() for v in (5, 6, 8)},
                     **{f'rug_prev_0.{v}': (c[(D.label.values == 1) & (D.solc_minor.values == v)] > 0).mean() for v in (5, 6, 8)}))
DT = pd.DataFrame(rows).sort_values(['impact', 'rug_mean'], ascending=[True, False])
DT['legit_range_across_solc'] = DT[[f'legit_prev_0.{v}' for v in (5, 6, 8)]].max(axis=1) - DT[[f'legit_prev_0.{v}' for v in (5, 6, 8)]].min(axis=1)
save('3a_detectors', DT.round(3))
print(DT[DT.impact.isin(['Optimization', 'Informational'])].round(2).to_string(index=False))

# composition of the optimization count
opt = DT[DT.impact == 'Optimization'][['check', 'rug_mean', 'legit_mean']].copy()
opt['share_of_rug_opt_count'] = opt.rug_mean / opt.rug_mean.sum(); opt['share_of_legit_opt_count'] = opt.legit_mean / opt.legit_mean.sum()
save('3b_optimization_composition', opt.round(3))

VERSION_DEP_PRIOR = {'solc-version', 'pragma', 'immutable-states'}        # stated before looking at the data
VERSION_DEP_EMP = set(DT[(DT.impact.isin(['Optimization', 'Informational'])) & (DT.legit_range_across_solc >= 0.25)].check)
print('a priori:', VERSION_DEP_PRIOR, '\nempirical (legit prevalence differs >= 0.25 across 0.5/0.6/0.8):', VERSION_DEP_EMP)

def feats_without(drop):
    return pd.DataFrame([slither_feats([x for x in DET[a] if x['check'] not in drop]) for a in D.address])

S = {'S0 all detectors (published)': D[FEAT].copy()}
for nm, drop in [('S1 without a-priori version-dependent detectors', VERSION_DEP_PRIOR),
                 ('S2 without a-priori + empirical version-dependent', VERSION_DEP_PRIOR | VERSION_DEP_EMP)]:
    X = D[FEAT].copy(); X[SL_COLS] = feats_without(drop)[SL_COLS].values; S[nm] = X
X = D[FEAT].copy(); X['total_vulnerabilities'] = D[['high_severity', 'medium_severity', 'low_severity']].sum(axis=1)
S['S3 optimization and informational counts removed'] = X.drop(columns=['optimization', 'informational'])
S['S4 no Slither counts (flags, regex, structure only)'] = D[[c for c in FEAT if c not in SL_COLS[:6]]]
rows, folds = [], {}
for nm, X in S.items():
    f, _ = cv_folds(X, y, G0); folds[nm] = f; rows.append({'set': nm} | summarise(f))
res3 = pd.DataFrame(rows)
tests = [paired_nb(folds[list(S)[0]], folds[nm]) for nm in list(S)[1:]]
res3['diff_vs_S0'] = [None] + [t[0] for t in tests]; res3['p_holm'] = [None] + list(holm([t[1] for t in tests]).round(4))
save('3c_version_sensitivity', res3); print(res3.to_string(index=False))

# SHAP ranking after removing version-dependent detectors
import shap
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
def shap_rank(X):
    X = X.reset_index(drop=True); sv = np.zeros(X.shape)
    for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, y, G0):
        m = xgb_model(0); m.fit(X.iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
        sv[te] = shap.TreeExplainer(m)(X.iloc[te]).values
    return pd.Series(np.abs(sv).mean(0), index=X.columns).sort_values(ascending=False)
top = {nm: shap_rank(S[nm]).head(10) for nm in [list(S)[0], list(S)[2]]}
save('3d_shap_top10_S0_vs_S2', pd.DataFrame({nm: [f'{k} ({v:.2f})' for k, v in s.items()] for nm, s in top.items()}))

## 4. Enriched metadata baseline (information available at deployment, unrelated to fraud)

In [ ]:
def meta_row(r):
    res = r['res']; cv = res.get('CompilerVersion', '') or ''
    m = re.search(r'(\d+)\.(\d+)\.(\d+)', cv)
    return dict(solc_patch=int(m.group(3)) if m else np.nan, solc_nightly=int('nightly' in cv),
                opt_used=int(str(res.get('OptimizationUsed', '0')) == '1'),
                opt_runs=np.log1p(float(res.get('Runs') or 0)),
                evm=(res.get('EVMVersion') or 'default').lower(), license=(res.get('LicenseType') or 'none').lower())
M = pd.DataFrame([meta_row(INFO[a]) for a in D.address])
for c in ['evm', 'license']:
    keep = M[c].value_counts(); M[c] = M[c].where(M[c].map(keep) >= 10, 'other')
save('4a_metadata_by_class', pd.concat([pd.crosstab(M[c], D.label.values, normalize='columns').round(3).assign(field=c)
                                        for c in ['evm', 'license', 'opt_used', 'solc_nightly']]).reset_index(), index=False)
Menc = pd.get_dummies(M, columns=['evm', 'license'], dtype=int)
Menc[['year', 'solc_minor', 'is_proxy']] = D[['year', 'solc_minor', 'is_proxy']].values
sets = {'M0 year + compiler minor (published baseline)': D[['year', 'solc_minor']],
        'M1 enriched metadata': Menc,
        'M2 code features (29)': D[FEAT],
        'M3 code features + enriched metadata': pd.concat([D[FEAT].reset_index(drop=True), Menc.reset_index(drop=True)], axis=1)}
fm = {nm: cv_folds(X, y, G0)[0] for nm, X in sets.items()}
r4 = pd.DataFrame([{'set': nm} | summarise(f) for nm, f in fm.items()])
k = list(fm); cmp = [(k[1], k[0]), (k[2], k[1]), (k[3], k[2])]
tt = [paired_nb(fm[a], fm[b]) for a, b in cmp]
save('4b_metadata_baseline', r4)
save('4c_metadata_tests', pd.DataFrame([dict(comparison=f'{a}  vs  {b}', diff_mcc=t[0], p_holm=p)
                                        for (a, b), t, p in zip(cmp, tt, holm([t[1] for t in tt]))]))
print(r4.to_string(index=False))

## 5. Duplicates and similarity on the source code itself

In [ ]:
SRCTXT = {a: '\n'.join(INFO[a]['files'].values()) for a in D.address}
N1 = {a: norm_level1(s) for a, s in SRCTXT.items()}; N2 = {a: norm_level2(s) for a, s in SRCTXT.items()}
rows = []
for lvl, N in [('L1 comments and whitespace removed', N1), ('L2 + literals replaced', N2)]:
    h = pd.Series({a: sha(N[a]) for a in D.address}).reindex(D.address).values
    t = pd.DataFrame({'h': h, 'label': y}); gsz = t.groupby('h').label.agg(['size', 'nunique'])
    for lab, nm in [(1, 'rug pull'), (0, 'legitimate')]:
        s = t[t.label == lab]
        rows.append(dict(level=lvl, cls=nm, contracts=len(s), distinct_hashes=s.h.nunique(),
                         contracts_with_exact_duplicate=int(s.h.map(gsz['size']).gt(1).sum()),
                         in_hash_groups_with_both_classes=int(s.h.map(gsz['nunique']).gt(1).sum())))
save('5a_exact_duplicates', pd.DataFrame(rows)); print(pd.DataFrame(rows).to_string(index=False))

t0 = time.time()
H = minhash_matrix([shingles(N2[a]) for a in D.address], n_perm=256)
J = est_jaccard(H); np.fill_diagonal(J, 0); print(f'similarity matrix in {time.time()-t0:.0f}s')
np.save(f'{LOCAL}/J.npy', J)

rows, GS = [], {}
for T in (0.7, 0.8, 0.9):
    uf = UnionFind(len(D)); ii, jj = np.where(np.triu(J >= T))
    for i, j in zip(ii, jj): uf.union(i, j)
    lab = uf.labels(); c = pd.DataFrame({'c': lab, 'y': y}).groupby('c').y.agg(['size', 'nunique'])
    uf2 = UnionFind(len(D))                                   # similarity clusters merged with the feature clusters
    for i, j in zip(ii, jj): uf2.union(i, j)
    first_of = {}
    for i, g in enumerate(G0):
        if g in first_of: uf2.union(first_of[g], i)
        else: first_of[g] = i
    GS[T] = uf2.labels()
    rows.append(dict(threshold=T, similarity_clusters=len(c), largest=int(c['size'].max()),
                     clusters_with_both_classes=int((c['nunique'] > 1).sum()),
                     legit_in_mixed_clusters=int(((pd.Series(lab).map(c['nunique']) > 1) & (y == 0)).sum()),
                     merged_groups_for_cv=len(set(GS[T]))))
save('5b_similarity_clusters', pd.DataFrame(rows)); print(pd.DataFrame(rows).to_string(index=False))

rows = [{'grouping': 'published (feature space)'} | summarise(F0)]
for T in (0.9, 0.8, 0.7):
    if len(set(GS[T])) < 25: print(f'threshold {T}: only {len(set(GS[T]))} groups, CV not possible'); continue
    f, _ = cv_folds(D[FEAT], y, GS[T]); rows.append({'grouping': f'feature clusters + source similarity >= {T}'} | summarise(f))
save('5c_cv_source_grouping', pd.DataFrame(rows)); print(pd.DataFrame(rows).to_string(index=False))

# temporal split: is detection explained by similarity to the training contracts?
tr = np.where(D.year.values <= 2021)[0]; te = np.where(D.year.values >= 2022)[0]
m = xgb_model(0); m.fit(D[FEAT].iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
pt = (m.predict_proba(D[FEAT].iloc[te])[:, 1] >= 0.5).astype(int)
mx = J[np.ix_(te, tr)].max(axis=1)
T5 = pd.DataFrame({'label': y[te], 'pred': pt, 'max_sim_to_train': mx})
T5['similarity_bin'] = pd.cut(T5.max_sim_to_train, [-0.01, 0.5, 0.7, 0.8, 0.9, 1.0])
t5 = T5.groupby(['label', 'similarity_bin'], observed=True).pred.agg(n='size', flagged_as_rug='mean').reset_index()
save('5d_temporal_by_similarity', t5.round(3)); print(t5.round(3).to_string(index=False))

# legitimate tokens that resemble rug pulls (used for the top-20 audit)
rug_idx = np.where(y == 1)[0]
D['max_sim_to_rug'] = J[:, rug_idx].max(axis=1)
D['nearest_rug'] = D.address.values[rug_idx][J[:, rug_idx].argmax(axis=1)]
lg = D[D.label == 0]
save('5e_legit_similarity_to_rugs', pd.DataFrame({'max similarity to any rug pull': ['>= 0.9', '>= 0.8', '>= 0.7', '< 0.5'],
     'legitimate tokens': [(lg.max_sim_to_rug >= .9).sum(), (lg.max_sim_to_rug >= .8).sum(), (lg.max_sim_to_rug >= .7).sum(), (lg.max_sim_to_rug < .5).sum()],
     'of which flagged (OOF)': [lg[lg.max_sim_to_rug >= t].pred_oof.sum() for t in (.9, .8, .7)] + [lg[lg.max_sim_to_rug < .5].pred_oof.sum()]}))

## 6. Regex indicators on different parts of the source

In [ ]:
NC = {a: strip_comments(s) for a, s in SRCTXT.items()}
V = {'V0 whole source (published)': lambda a: SRCTXT[a],
     'V1 comments removed': lambda a: NC[a],
     'V2 main contract only': lambda a: main_contract_body(NC[a], INFO[a]['name'])[0],
     'V3 project contracts (no interfaces, libraries, standard contracts)': lambda a: project_code(NC[a])}
RXV, miss = {}, {}
for nm, fn in V.items():
    rows = []
    for a in D.address:
        t = fn(a); miss.setdefault(nm, 0)
        if not t: miss[nm] += 1; t = NC[a]
        rows.append(regex_feats(t))
    RXV[nm] = pd.DataFrame(rows)
print('fallback to whole source (main contract not found):', miss)
if PARITY_OK: assert (RXV[list(V)[0]].values == D[RX_COLS].values).all(), 'V0 must equal the published regex features'

prev = []
for nm, R in RXV.items():
    t = R.assign(group=D.group.values).groupby('group').mean().T; t.insert(0, 'variant', nm); prev.append(t)
save('6a_regex_prevalence', pd.concat(prev).round(3).reset_index().rename(columns={'index': 'indicator'}))
mv = pd.DataFrame([mint_variants(project_code(NC[a])) for a in D.address]).assign(group=D.group.values).groupby('group').mean()
save('6b_mint_function_vs_constructor', mv.round(3).reset_index())

rows, H2 = [], []
f_noregex, _ = cv_folds(D[[c for c in FEAT if c not in RX_COLS]], y, G0)
for nm, R in RXV.items():
    X = D[FEAT].copy(); X[RX_COLS] = R.values
    f_all, _ = cv_folds(X, y, G0); f_rx, _ = cv_folds(R, y, G0)
    d_, p_ = paired_nb(f_all, f_noregex)
    rows.append(dict(variant=nm, mcc_all29=summarise(f_all)['mcc'], mcc_regex_only=summarise(f_rx)['mcc'],
                     all29_minus_without_regex=d_, p=p_))
r6 = pd.DataFrame(rows); r6['p_holm'] = holm(r6.p).round(4)
save('6c_regex_variants_cv', r6); print(r6.to_string(index=False))

In [ ]:
# 7. V4: the 12 regex patterns with ONE fix only: underscore and $ count as word separators.
#    Python's \b treats '_' as a letter, so \bmaxtxamount\b cannot match "_maxTxAmount".
#    No keyword is added or removed. Paste below section 6 (needs D, SRCTXT, NC, INFO, FEAT, RX_COLS,
#    y, G0, GS, cv_folds, summarise, paired_nb, holm, save, main_contract_body). Runtime ~10 min.
import re, inspect
import rev2lib

# 1. Copy the published patterns straight from rev2lib.text_feats, so nothing is retyped by hand.
PATS = dict(re.findall(r"(has_\w+)=rx\(r'([^']+)'\)", inspect.getsource(rev2lib.text_feats)))
assert list(PATS) == RX_COLS, (list(PATS), RX_COLS)

def fix_boundary(p):
    """Replace each \\b at the edge of an alternative by lookarounds that only treat [a-z0-9] as word chars."""
    alts = []
    for a in p.split('|'):
        core = a.replace(r'\b', '')
        alts.append(('(?<![a-z0-9])' if a.startswith(r'\b') else '') + core + ('(?![a-z0-9])' if a.endswith(r'\b') else ''))
    return '|'.join(alts)

P0 = {k: re.compile(v) for k, v in PATS.items()}               # published
P4 = {k: re.compile(fix_boundary(v)) for k, v in PATS.items()}  # V4
for k in RX_COLS: print(f'{k:18s} {PATS[k]}\n{"":18s} {P4[k].pattern}')

def flags(text, P): s = text.lower(); return {k: int(bool(P[k].search(s))) for k in RX_COLS}

# 2. Parity guard: the copied patterns must reproduce the published features exactly.
R0 = pd.DataFrame([flags(SRCTXT[a], P0) for a in D.address])
assert (R0.values == D[RX_COLS].values).all(), 'copied patterns do not reproduce the published regex features'

# 3. V4 on the whole source (compare with V0) and on the main contract (compare with V2).
def main_or_nc(a):
    t = main_contract_body(NC[a], INFO[a]['name'])[0]
    return t if t else NC[a]
R4 = pd.DataFrame([flags(SRCTXT[a], P4) for a in D.address])
R4m = pd.DataFrame([flags(main_or_nc(a), P4) for a in D.address])

# 4. How many contracts change, per indicator and direction.
ch = pd.DataFrame({'indicator': RX_COLS,
                   'V0_to_V4_0to1': [int(((R0[c] == 0) & (R4[c] == 1)).sum()) for c in RX_COLS],
                   'V0_to_V4_1to0': [int(((R0[c] == 1) & (R4[c] == 0)).sum()) for c in RX_COLS]})
save('7a_V4_changed_contracts', ch); print(ch.to_string(index=False))

prev = []
for nm, R in [('V0 whole source (published)', R0), ('V4 whole source, boundary fixed', R4), ('V4 main contract, boundary fixed', R4m)]:
    t = R.assign(group=D.group.values).groupby('group').mean().T; t.insert(0, 'variant', nm); prev.append(t)
save('7b_V4_prevalence', pd.concat(prev).round(3).reset_index().rename(columns={'index': 'indicator'}))

# 5. Cross-validation, same protocol and grouping as sheet 6c (feature clusters, G0).
f_noregex, _ = cv_folds(D[[c for c in FEAT if c not in RX_COLS]], y, G0)
X0 = D[FEAT].copy(); f_all0, _ = cv_folds(X0, y, G0)
rows = []
for nm, R in [('V4 whole source, boundary fixed', R4), ('V4 main contract, boundary fixed', R4m)]:
    X = D[FEAT].copy(); X[RX_COLS] = R.values
    f_all, _ = cv_folds(X, y, G0); f_rx, _ = cv_folds(R, y, G0)
    d1, p1 = paired_nb(f_all, f_noregex); d2, p2 = paired_nb(f_all, f_all0)
    rows.append(dict(variant=nm, mcc_all29=summarise(f_all)['mcc'], mcc_regex_only=summarise(f_rx)['mcc'],
                     all29_minus_without_regex=d1, p_vs_without_regex=p1, all29_minus_published=d2, p_vs_published=p2))
r7 = pd.DataFrame(rows)
r7['p_holm_vs_without_regex'] = holm(r7.p_vs_without_regex).round(4)
r7['p_holm_vs_published'] = holm(r7.p_vs_published).round(4)
save('7c_V4_cv', r7); print(r7.to_string(index=False))

# 6. Flags per address, for checking V4 against the manual annotation outside Colab.
out = pd.concat([D[['address']].reset_index(drop=True),
                 R4.add_prefix('V4_'), R4m.add_prefix('V4m_')], axis=1)
out.to_csv(f'{OUT}/regex_V4_flags.csv', index=False)
print('wrote', f'{OUT}/regex_V4_flags.csv', out.shape)

## 7. Files for the manual work

In [ ]:
rng = np.random.default_rng(2026)
plan = {'Rug: Transaction Limitation': 20, 'Rug: Fee': 15, 'Rug: other or CRPWarner only': 10, 'Legitimate': 15}
samp = pd.concat([D[D.group == g].sample(n, random_state=2026) for g, n in plan.items()]).sample(frac=1, random_state=2026)
samp['id'] = [f'C{i:02d}' for i in range(1, len(samp) + 1)]
for _, r in samp.iterrows():
    with open(f'{OUT}/annotation/contracts/{r.id}.sol', 'w') as fh:
        fh.write(f'// Annotation file {r.id}. Main contract: {INFO[r.address]["name"]}\n' + SRCTXT[r.address])
MECH = ['sell_restriction', 'max_tx_or_wallet_limit', 'blacklist', 'pause', 'fee_changeable_by_owner', 'whitelist_or_fee_exclusion',
        'transfer_lock_or_cooldown', 'mint_after_deployment', 'mint_in_constructor_only', 'owner_only_functions',
        'renounce_ownership', 'burn', 'selfdestruct', 'proxy_or_delegatecall']
sheet = samp[['id']].copy(); sheet['main_contract'] = [INFO[a]['name'] for a in samp.address]
for c in MECH: sheet[c] = ''
sheet['annotator'] = ''; sheet['minutes_spent'] = ''; sheet['notes'] = ''
codebook = pd.DataFrame([
 ('General', 'Judge only code written for this token (the main contract and its own parent contracts). Ignore interfaces, SafeMath, Context and unmodified standard OpenZeppelin code. Answer Y, N, or ? if you cannot decide.'),
 ('sell_restriction', 'Owner-controlled logic can stop or limit selling, e.g. transfers to the Uniswap pair/router require a flag, a list, or a condition the owner sets.'),
 ('max_tx_or_wallet_limit', 'A maximum amount per transaction or per wallet is enforced in a transfer.'),
 ('blacklist', 'Specific addresses can be blocked from transferring, by a list or mapping the owner can change.'),
 ('pause', 'A global switch can stop all transfers.'),
 ('fee_changeable_by_owner', 'A transfer fee or tax exists AND the owner can change its value after deployment.'),
 ('whitelist_or_fee_exclusion', 'Some addresses are exempt from limits or fees.'),
 ('transfer_lock_or_cooldown', 'Transfers are blocked until a time, block, or trading-enabled flag, or a cooldown applies between trades.'),
 ('mint_after_deployment', 'A function callable after deployment can increase total supply or a balance without a matching decrease elsewhere.'),
 ('mint_in_constructor_only', 'Tokens are created only once, in the constructor.'),
 ('owner_only_functions', 'At least one function can be called only by the owner or an admin role.'),
 ('renounce_ownership', 'A function exists that removes the owner. Note in "notes" if owner powers remain through another role.'),
 ('burn', 'A function destroys tokens.'),
 ('selfdestruct', 'selfdestruct (or suicide) is called.'),
 ('proxy_or_delegatecall', 'The contract delegates calls to other code (delegatecall, upgradeable proxy).')], columns=['item', 'definition'])
with pd.ExcelWriter(f'{OUT}/annotation/regex_annotation_sheet.xlsx') as w:
    codebook.to_excel(w, sheet_name='codebook', index=False); sheet.to_excel(w, sheet_name='annotate', index=False)
key = samp[['id', 'address', 'label', 'group', 'Type']].copy()
for nm, R in RXV.items():
    for c in RX_COLS: key[f'{nm[:2]}_{c}'] = R.loc[samp.index, c].values
key.to_csv(f'{OUT}/annotation/annotation_key_DO_NOT_OPEN.csv', index=False)
print('annotation files written:', len(samp))

# top-20 legitimate tokens by out-of-fold score
fp = D[(D.label == 0) & (D.pred_oof == 1)].sort_values('p_oof', ascending=False).head(20).copy()
try:
    import requests
    cg = requests.get('https://api.coingecko.com/api/v3/coins/list', params={'include_platform': 'true'}, timeout=60).json()
    cgm = {c['platforms']['ethereum'].lower(): (c['id'], c['symbol']) for c in cg if (c.get('platforms') or {}).get('ethereum')}
except Exception as e:
    print('CoinGecko unavailable:', e); cgm = {}
leg886 = pd.read_csv(PATHS['legit886']); leg886['address'] = leg886.address.str.lower()
fp = fp.merge(leg886[['address', 'life_days']], on='address', how='left')
audit = pd.DataFrame({'address': fp.address, 'coingecko_id': [cgm.get(a, ('', ''))[0] for a in fp.address],
    'symbol': [cgm.get(a, ('', ''))[1] for a in fp.address], 'contract_name': [INFO[a]['name'] for a in fp.address],
    'year': fp.year, 'life_days': fp.life_days.round(0), 'p_rug_oof': fp.p_oof.round(3),
    'max_similarity_to_a_rug_pull': fp.max_sim_to_rug.round(2), 'nearest_rug_pull': fp.nearest_rug,
    'has_max_tx': fp.has_max_tx, 'has_fee_modifier': fp.has_fee_modifier, 'has_blacklist': fp.has_blacklist,
    'etherscan': 'https://etherscan.io/token/' + fp.address, 'honeypot_is': 'https://honeypot.is/ethereum?address=' + fp.address,
    'still_traded_2026 (Y/N)': '', 'scam_reports_found (Y/N + link)': '', 'honeypot_result': '',
    'verdict (legit / suspicious / likely scam)': '', 'notes': ''})
audit.to_excel(f'{OUT}/annotation/top20_legit_audit.xlsx', index=False)

# missed rug pulls: code excerpts
fn = D[(D.label == 1) & (D.pred_oof == 0)].sort_values('p_oof').copy()
HINT = re.compile(r'uniswapv2pair|\bpair\b|router|\bsell|tradingopen|tradingenabled|cooldown|\bbots?\b|blacklist|maxtx|maxwallet|_istxlimit|onlyowner', re.I)
lines_md = ['# Missed rug pulls (out-of-fold, threshold 0.5): code excerpts\n']
rows = []
for k, (_, r) in enumerate(fn.iterrows(), 1):
    a = r.address; src = NC[a]; code = project_code(src) or src
    others = np.setdiff1d(rug_idx, [D.index.get_loc(_)])
    rows.append(dict(id=f'F{k:02d}', address=a, type=r.Type, source=r.Sources, p_rug=round(r.p_oof, 3), year=r.year,
                     solc_minor=r.solc_minor, max_sim_to_other_rugs=round(float(J[D.index.get_loc(_), others].max()), 2),
                     **{c: r[c] for c in ['optimization', 'func_count', 'code_length', 'has_max_tx', 'has_fee_modifier', 'has_blacklist']}))
    lines_md.append(f'\n## F{k:02d}  type: {r.Type}  p={r.p_oof:.3f}  main contract: {INFO[a]["name"]}\n')
    for nm, ln, body in function_bodies(code, {'_transfer', 'transfer', 'transferFrom', '_tokenTransfer', '_beforeTokenTransfer'})[:3]:
        lines_md.append(f'\n`{nm}` (line {ln})\n```solidity\n' + '\n'.join(body.splitlines()[:60]) + '\n```\n')
    hits = [f'{i+1}: {l.strip()}' for i, l in enumerate(src.splitlines()) if HINT.search(l) and l.strip()][:25]
    lines_md.append('\nLines with sale-restriction hints:\n```\n' + '\n'.join(hits) + '\n```\n')
pd.DataFrame(rows).to_excel(f'{OUT}/annotation/missed_rugpulls.xlsx', index=False)
open(f'{OUT}/annotation/missed_rugpulls_excerpts.md', 'w').write('\n'.join(lines_md))
print('missed rug pulls exported:', len(fn))

## 8. Legitimate tokens with 180–365 days of activity
Needs your Etherscan key stored as a Colab secret named `ETHERSCAN_API_KEY`
(key icon in the left bar). Never paste the key into a cell.

In [ ]:
cs = pd.read_csv(PATHS['cand_span']).drop_duplicates('address'); cs['address'] = cs.address.str.lower()
cs['year'] = pd.to_datetime(cs.first_ts, unit='s').dt.year; cs['life_days'] = (cs.last_ts - cs.first_ts) / 86400
extra = cs[cs.year.between(2020, 2023) & (cs.life_days >= 180) & (cs.life_days < 365)]
print('candidates with 180-365 days:', len(extra), '| with >= 365 days:', int((cs.year.between(2020, 2023) & (cs.life_days >= 365)).sum()))
if not DRY_RUN and len(extra):
    from google.colab import userdata
    import requests
    KEY = userdata.get('ETHERSCAN_API_KEY')
    def fetch(addr):
        p = f'{RAW}/{addr}.json'
        if os.path.exists(p): return json.load(open(p))
        for _ in range(4):
            try:
                r = requests.get('https://api.etherscan.io/v2/api', timeout=30, params=dict(chainid=1, module='contract',
                                 action='getsourcecode', address=addr, apikey=KEY)).json()
                if isinstance(r.get('result'), list) and r['result']:
                    json.dump(r['result'][0], open(p, 'w')); time.sleep(0.22); return r['result'][0]
            except Exception: pass
            time.sleep(1.5)
        return None
    new = []
    for a in extra.address:
        res = fetch(a)
        if res and res.get('Proxy') == '1' and res.get('Implementation'): fetch(res['Implementation'].lower())
        res, px = eff_res(RAW, a); u = unpack(res) if res else None
        if not u: new.append(dict(address=a, status='no_verified_source')); continue
        files, remaps = u
        if len(files) > 1: new.append(dict(address=a, status='multi_file')); continue
        if not E20 <= abi_names(res): new.append(dict(address=a, status='not_erc20')); continue
        v = solc_ver(res); subprocess.run(['solc-select', 'install', v], capture_output=True)
        det, st = run_slither_json(write_tree(f'{LOCAL}/src', a, files), pick_main(files, res.get('ContractName')), v, remaps)
        if st != 'ok': new.append(dict(address=a, status=st)); continue
        rec = dict(address=a, status='ok', label=0, is_proxy=px, solc_minor=solc_minor(res))
        rec.update(slither_feats(det)); rec.update(text_feats('\n'.join(files.values()))); new.append(rec)
    new = pd.DataFrame(new).merge(extra[['address', 'year', 'life_days']], on='address')
    print(new.status.value_counts())
    ok = new[new.status == 'ok']
    rows = [{'legitimate class': 'published (>= 365 days)', 'n_legit': int((y == 0).sum())} | summarise(F0)]
    if len(ok):
        D8 = pd.concat([D[['address', 'label', 'year'] + FEAT], ok[['address', 'label', 'year'] + FEAT]]).reset_index(drop=True)
        f8, _ = cv_folds(D8[FEAT], D8.label.values, feature_groups(D8))
        rows.append({'legitimate class': '>= 180 days (adds tokens with 180-365 days)', 'n_legit': int((D8.label == 0).sum())} | summarise(f8))
    save('8_activity_threshold_180', pd.concat([pd.DataFrame(rows), new.drop(columns=FEAT, errors='ignore')], ignore_index=True))

## 9. Score the annotation (run only after `regex_annotation_sheet.xlsx` is filled in)

In [ ]:
ann = pd.read_excel(f'{OUT}/annotation/regex_annotation_sheet.xlsx', sheet_name='annotate')
key = pd.read_csv(f'{OUT}/annotation/annotation_key_DO_NOT_OPEN.csv')
A = ann.merge(key, on='id')
MAP = {'has_max_tx': 'max_tx_or_wallet_limit', 'has_blacklist': 'blacklist', 'has_pause': 'pause',
       'has_fee_modifier': 'fee_changeable_by_owner', 'has_whitelist': 'whitelist_or_fee_exclusion',
       'has_transfer_lock': 'transfer_lock_or_cooldown', 'has_mint': 'mint_after_deployment', 'has_owner_only': 'owner_only_functions',
       'has_renounce': 'renounce_ownership', 'has_burn': 'burn', 'has_selfdestruct': 'selfdestruct', 'has_proxy': 'proxy_or_delegatecall'}
rows = []
for rx_c, mech in MAP.items():
    truth = A[mech].astype(str).str.upper().str.strip().map({'Y': 1, 'N': 0})
    for v in ['V0', 'V1', 'V2', 'V3']:
        pr = A[f'{v}_{rx_c}']; ok = truth.notna()
        tp = int(((pr == 1) & (truth == 1) & ok).sum()); fp_ = int(((pr == 1) & (truth == 0) & ok).sum()); fn_ = int(((pr == 0) & (truth == 1) & ok).sum())
        rows.append(dict(indicator=rx_c, variant=v, n_annotated=int(ok.sum()), present_by_annotation=int((truth == 1).sum()),
                         precision=tp / (tp + fp_) if tp + fp_ else np.nan, recall=tp / (tp + fn_) if tp + fn_ else np.nan))
save('9a_regex_validity', pd.DataFrame(rows).round(3))
sr = A.assign(sell=A.sell_restriction.astype(str).str.upper().str.strip()).groupby('group').sell.value_counts().unstack(fill_value=0)
save('9b_sell_restriction_by_group', sr.reset_index())
if A.annotator.nunique() > 1:
    from sklearn.metrics import cohen_kappa_score
    print('Two annotators found: add the double-coded rows and compute kappa per mechanism here.')

In [ ]:
# 5f. How robust is the source-similarity result?  (paste into a new cell after Section 5 and run)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import matthews_corrcoef
X = D[FEAT].reset_index(drop=True)

# (a) the largest similarity family at threshold 0.9 (similarity only)
uf = UnionFind(len(D)); ii, jj = np.where(np.triu(J >= 0.9))
for i, j in zip(ii, jj): uf.union(i, j)
lab09 = uf.labels(); vc = pd.Series(lab09).value_counts()
big = np.where(lab09 == vc.index[0])[0]
pair = J[np.ix_(big, big)][np.triu_indices(len(big), 1)]
fam = dict(size=len(big), rug_pulls=int(y[big].sum()), legitimate=int((y[big] == 0).sum()),
           median_pairwise_similarity=round(float(np.median(pair)), 3), p10_pairwise_similarity=round(float(np.percentile(pair, 10)), 3),
           share_of_pairs_at_least_0_9=round(float((pair >= 0.9).mean()), 3),
           second_largest=int(vc.iloc[1]), third_largest=int(vc.iloc[2]),
           groups=str(D.group.values[big].tolist().count('Rug: Transaction Limitation')) + ' TL / ' +
                  str(D.group.values[big].tolist().count('Rug: Fee')) + ' Fee',
           years=str(pd.Series(D.year.values[big]).value_counts().sort_index().to_dict()),
           solc=str(pd.Series(D.solc_minor.values[big]).value_counts().sort_index().to_dict()))
save('5f1_largest_family', pd.DataFrame([fam]).T.reset_index().rename(columns={'index': 'item', 0: 'value'}))

# (b) fold-by-fold results of the similarity-grouped CV (threshold 0.9)
rows = []
for rep in range(5):
    for k, (tr, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=rep).split(X, y, GS[0.9])):
        if len(set(y[tr])) < 2 or len(set(y[te])) < 2: continue
        m = xgb_model(rep); m.fit(X.iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
        pr = (m.predict_proba(X.iloc[te])[:, 1] >= 0.5).astype(int)
        rows.append(dict(rep=rep, fold=k, test_rug=int(y[te].sum()), test_legit=int((y[te] == 0).sum()),
                         train_rug=int(y[tr].sum()), largest_family_in_test=bool(np.isin(big, te).mean() > 0.5),
                         mcc=round(matthews_corrcoef(y[te], pr), 3), recall=round(pr[y[te] == 1].mean(), 3),
                         specificity=round(1 - pr[y[te] == 0].mean(), 3)))
f5 = pd.DataFrame(rows); save('5f2_similarity_cv_by_fold', f5)
print(f5.groupby('largest_family_in_test')[['mcc', 'recall', 'specificity', 'test_rug', 'train_rug']].mean().round(3))

# (c) conservative grouping: exact duplicates only (normalised hash) merged with the feature clusters
h2 = pd.Series([sha(N2[a]) for a in D.address]).factorize()[0]
uf = UnionFind(len(D)); first = {}
for i, (g1, g2) in enumerate(zip(G0, h2)):
    for key in (('f', g1), ('h', g2)):
        if key in first: uf.union(first[key], i)
        else: first[key] = i
Gh = uf.labels(); fh, _ = cv_folds(X, y, Gh)
save('5f3_exact_duplicate_grouping', pd.DataFrame([{'grouping': 'feature clusters + exact normalised duplicates',
                                                    'groups': len(set(Gh))} | summarise(fh)]))

# (d) train without the largest family, test on it; and the reverse
out_ = np.setdiff1d(np.arange(len(D)), big)
rows = []
for seed in range(5):
    m = xgb_model(seed); m.fit(X.iloc[out_], y[out_], sample_weight=compute_sample_weight('balanced', y[out_]))
    p = (m.predict_proba(X.iloc[big])[:, 1] >= 0.5).astype(int)
    rows.append(dict(direction='trained without the largest family, tested on it', seed=seed,
                     recall_rug=round(p[y[big] == 1].mean(), 3), n_rug=int(y[big].sum())))
    leg = np.where(y == 0)[0]; rng_ = np.random.default_rng(seed); te_leg = rng_.choice(np.setdiff1d(leg, big), size=len(leg) // 5, replace=False)
    tr2 = np.union1d(big, np.setdiff1d(np.setdiff1d(leg, big), te_leg)); te2 = np.union1d(np.setdiff1d(out_, leg), te_leg)
    m = xgb_model(seed); m.fit(X.iloc[tr2], y[tr2], sample_weight=compute_sample_weight('balanced', y[tr2]))
    p = (m.predict_proba(X.iloc[te2])[:, 1] >= 0.5).astype(int)
    rows.append(dict(direction='trained on the largest family + legit, tested on other families', seed=seed,
                     recall_rug=round(p[y[te2] == 1].mean(), 3), n_rug=int(y[te2].sum()),
                     specificity=round(1 - p[y[te2] == 0].mean(), 3)))
f6 = pd.DataFrame(rows); save('5f4_family_transfer', f6)
print(f6.groupby('direction')[['recall_rug']].agg(['mean', 'std']).round(3))

# (e) identical or near-identical sources across the two classes
h1 = pd.Series([sha(N1[a]) for a in D.address])
cross = h1[h1.map(pd.Series(y).groupby(h1).nunique()) > 1]
rows = [dict(kind='identical source (L1)', address=D.address[i], label=int(y[i]), group=D.group[i], p_oof=round(D.p_oof[i], 3)) for i in cross.index]
li = np.where(y == 0)[0]; ri = np.where(y == 1)[0]
for i in li[np.argsort(-J[np.ix_(li, ri)].max(axis=1))][:15]:
    rows.append(dict(kind='legitimate token closest to a rug pull', address=D.address[i], label=0, group=D.group[i],
                     p_oof=round(D.p_oof[i], 3), similarity=round(float(J[i, ri].max()), 3), nearest_rug=D.address[ri[J[i, ri].argmax()]]))
save('5f5_cross_class_near_duplicates', pd.DataFrame(rows)); print(pd.DataFrame(rows).head(20).to_string(index=False))

In [ ]:
# 5g. Re-test the paper's hypotheses with source-similarity grouping (paste below cell 5f and run, ~15 min)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import matthews_corrcoef, average_precision_score, roc_auc_score

GRP = GS[0.9]                      # feature clusters + source similarity >= 0.9 (same groups as sheet 5c)
def models(seed):                  # identical settings to the published pipeline
    return {'LogReg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced', random_state=seed)),
            'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed, n_jobs=-1),
            'XGBoost': xgb_model(seed)}
def cv_g(X, only=None):
    X = pd.DataFrame(X).reset_index(drop=True); rows = []
    for rep in range(5):
        for k, (tr, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=rep).split(X, y, GRP)):
            if len(set(y[te])) < 2 or len(set(y[tr])) < 2: continue
            for nm, m in models(rep).items():
                if only and nm != only: continue
                if nm == 'XGBoost': m.fit(X.iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
                else: m.fit(X.iloc[tr], y[tr])
                p = m.predict_proba(X.iloc[te])[:, 1]; pr = (p >= 0.5).astype(int)
                rows.append(dict(model=nm, rep=rep, fold=k, mcc=matthews_corrcoef(y[te], pr), spec=1 - pr[y[te] == 0].mean(),
                                 rec=pr[y[te] == 1].mean(), pr_auc_legit=average_precision_score(1 - y[te], 1 - p),
                                 roc_auc=roc_auc_score(y[te], p), big_in_test=bool(np.isin(big, te).mean() > 0.5)))
    return pd.DataFrame(rows)
def summ(f):
    s = summarise(f); s['mcc_folds_without_largest_family'] = f'{f[~f.big_in_test].mcc.mean():.3f} ± {f[~f.big_in_test].mcc.std():.3f}'
    return s

# (a) models
fx = cv_g(D[FEAT]); XG = fx[fx.model == 'XGBoost']
r = pd.DataFrame([{'model': m} | summ(fx[fx.model == m]) for m in ['LogReg', 'RandomForest', 'XGBoost']])
tests = [(f'XGBoost vs {o}', met, *paired_nb(XG, fx[fx.model == o], met)) for o in ['RandomForest', 'LogReg'] for met in ['mcc', 'pr_auc_legit']]
t = pd.DataFrame(tests, columns=['comparison', 'metric', 'diff', 'p']); t['p_holm'] = holm(t.p).round(4)
save('5g1_models_similarity_grouping', r); save('5g2_model_tests_similarity_grouping', t)
print(r.to_string(index=False)); print(t.to_string(index=False))

# (b) H1: code versus metadata
sets = {'M0 year + compiler minor': D[['year', 'solc_minor']], 'M1 enriched metadata': Menc,
        'M3 code + enriched metadata': pd.concat([D[FEAT].reset_index(drop=True), Menc.reset_index(drop=True)], axis=1)}
fm = {k: cv_g(v, only='XGBoost') for k, v in sets.items()}
r = pd.DataFrame([{'set': 'M2 code features (29)'} | summ(XG)] + [{'set': k} | summ(f) for k, f in fm.items()])
cmp = [('M2 code vs M0', XG, fm['M0 year + compiler minor']), ('M2 code vs M1 enriched metadata', XG, fm['M1 enriched metadata']),
       ('M3 code+meta vs M2 code', fm['M3 code + enriched metadata'], XG)]
t = pd.DataFrame([(n, *paired_nb(a, b)) for n, a, b in cmp], columns=['comparison', 'diff_mcc', 'p']); t['p_holm'] = holm(t.p).round(4)
save('5g3_H1_similarity_grouping', r); save('5g4_H1_tests_similarity_grouping', t)
print(r.to_string(index=False)); print(t.to_string(index=False))

# (c) H2 and feature-group ablation
sl_sev = SL_COLS[:6]; sl_flag = SL_COLS[6:]
V2 = RXV['V2 main contract only'].set_axis([c + '_v2' for c in RX_COLS], axis=1)
abl = {'without Slither': D[RX_COLS + ST_COLS], 'without regex': D[sl_sev + sl_flag + ST_COLS],
       'without structural': D[sl_sev + sl_flag + RX_COLS], 'Slither only': D[sl_sev + sl_flag], 'regex only': D[RX_COLS],
       'structural only': D[ST_COLS], 'regex only (main contract)': V2,
       'all 29 with main-contract regex': pd.concat([D[sl_sev + sl_flag + ST_COLS].reset_index(drop=True), V2], axis=1)}
rows = []
for k, X in abl.items():
    f = cv_g(X, only='XGBoost'); d_, p_ = paired_nb(XG, f)
    rows.append({'set': k} | summ(f) | {'all29_minus_this': d_, 'p': p_})
a = pd.DataFrame(rows); a['p_holm'] = holm(a.p).round(4)
save('5g5_ablation_similarity_grouping', a); print(a.to_string(index=False))

In [ ]:
# 10. Evidence + draft verdicts for the audit of the 20 highest-scoring legitimate tokens (reviewer A, round 2).
#     Collects facts only: last transfer (Etherscan), sell simulation (honeypot.is), contract risk flags (GoPlus),
#     market activity (CoinGecko). Your filled sheet is NOT touched; output goes to a new file.
#     Needs the Colab secret ETHERSCAN_API_KEY (never paste the key into a cell). Runtime ~3 min.
import time, json, requests
from google.colab import userdata
KEY = userdata.get('ETHERSCAN_API_KEY')
AUD = pd.read_excel(f'{OUT}/annotation/top20_legit_audit.xlsx')
AUD['address'] = AUD.address.str.lower()
CUTOFF = int(pd.Timestamp('2026-01-01', tz='UTC').timestamp())

def get(url, params=None, tries=3, pause=1.5):
    for _ in range(tries):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200: return r.json()
            if r.status_code == 429: time.sleep(10)
        except Exception: pass
        time.sleep(pause)
    return None

def dig(d, *ks):
    for k in ks:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

rows = []
for _, t in AUD.iterrows():
    a = t.address; r = {'address': a, 'contract_name': t.contract_name}
    # Etherscan: most recent transfers of this token
    es = get('https://api.etherscan.io/v2/api', dict(chainid=1, module='account', action='tokentx', contractaddress=a,
                                                     page=1, offset=100, sort='desc', apikey=KEY))
    tx = es.get('result') if es and isinstance(es.get('result'), list) else []
    ts = [int(x['timeStamp']) for x in tx]
    r['last_transfer_utc'] = pd.to_datetime(max(ts), unit='s').strftime('%Y-%m-%d') if ts else None
    r['transfers_since_2026_01_01 (max 100)'] = sum(s >= CUTOFF for s in ts)
    time.sleep(0.25)
    # honeypot.is: buy/sell simulation against the current main pair
    hp = get('https://api.honeypot.is/v2/IsHoneypot', dict(address=a, chainID=1))
    r['hp_simulation_ok'] = dig(hp, 'simulationSuccess')
    r['hp_is_honeypot'] = dig(hp, 'honeypotResult', 'isHoneypot')
    r['hp_buy_tax_%'] = dig(hp, 'simulationResult', 'buyTax')
    r['hp_sell_tax_%'] = dig(hp, 'simulationResult', 'sellTax')
    r['hp_liquidity_usd'] = dig(hp, 'pair', 'liquidity')
    r['hp_risk'] = dig(hp, 'summary', 'risk')
    r['hp_flags'] = json.dumps(dig(hp, 'summary', 'flags'))[:300] if hp else None
    r['hp_error'] = dig(hp, 'simulationError')
    # GoPlus: static and dynamic risk flags of the token contract
    gp = get(f'https://api.gopluslabs.io/api/v1/token_security/1', dict(contract_addresses=a))
    g = dig(gp, 'result', a) or {}
    for k in ['is_honeypot', 'cannot_sell_all', 'buy_tax', 'sell_tax', 'transfer_pausable', 'is_blacklisted',
              'owner_change_balance', 'hidden_owner', 'is_mintable', 'owner_address', 'holder_count', 'is_in_dex']:
        r[f'gp_{k}'] = g.get(k)
    time.sleep(1.0)
    # CoinGecko: is there still a market? (only tokens with a CoinGecko id)
    if isinstance(t.coingecko_id, str):
        cg = get(f'https://api.coingecko.com/api/v3/coins/{t.coingecko_id}',
                 dict(localization='false', tickers='false', community_data='false', developer_data='false'), pause=6)
        r['cg_volume_24h_usd'] = dig(cg, 'market_data', 'total_volume', 'usd')
        r['cg_market_cap_usd'] = dig(cg, 'market_data', 'market_cap', 'usd')
        r['cg_last_updated'] = dig(cg, 'last_updated')
        time.sleep(6)                      # free CoinGecko API allows about 10 calls per minute
    rows.append(r); print(a[:10], r['last_transfer_utc'], r['hp_is_honeypot'], r['gp_sell_tax'])

EV = pd.DataFrame(rows)

# Web search for scam reports (done by Claude, 23 Nov 2026). 'Y?' = only a low-credibility source.
SCAM = {
 '0xe03b2642a5111ad0efc0cbce766498c2dd562ae9': ('N', '', 'no web information found (address search)'),
 '0x1f36fb2d91d9951cf58ae4c1956c0b77e224f1e9': ('N', 'https://vcgamers.com/news/en/?p=68178', 'deployer wallet compromised by a hacker on 8 Nov 2022 (official statement); not a rug pull report'),
 '0x24ccedebf841544c9e6a62af4e8c2fa6e5a46fde': ('N', 'https://tokeninsight.com/en/coins/bluesparrow/rating', 'token of the BlueBit exchange'),
 '0x83249c6794bca5a77eb8c0af9f1a86e055459cea': ('N', 'https://coingecko.com/en/coins/gigaswap', 'GigaSwap OTC token; price about -99% from ATH, very low volume'),
 '0x53fd2342b43ecd24aef1535bc3797f509616ce8c': ('N', 'https://www.dextools.io/app/ether/pair-explorer/0x42692724953ec6add159979d3f969ef03b324c54', 'DEXTools audit: honeypot no, tax 0%, ownership renounced'),
 '0x9b2b931d6ab97b6a887b2c5d8529537e6fe73ebe': ('N', '', 'no web information found (address search)'),
 '0x1ae7e1d0ce06364ced9ad58225a1705b3e5db92b': ('Y?', 'https://cryptolinks.com/2822/lmeow', 'directory text says the original dev pulled the rug and the community continued; LP locked on Unicrypt (cyberscope)'),
 '0xe4b4c008ff36e3c50c4299c223504a480de9c833': ('N', '', 'no web information found (address search)'),
 '0x1e2f15302b90edde696593607b6bd444b64e8f02': ('N', 'https://coinranking.com/coin/kKjSOlEzF+shiryo-inu-shiryoinu/about', '11% transaction tax by design'),
 '0x9f4909cc95fb870bf48c128c1fdbb5f482797632': ('N', 'https://www.coingecko.com/en/coins/guzzler', ''),
 '0x2559813bbb508c4c79e9ccce4703bcb1f149edd7': ('N', 'https://blockspot.io/coin/hourglass/', 'project website offline since 2026-06-07'),
 '0x9e18d5bab2fa94a6a95f509ecb38f8f68322abd3': ('N', 'https://coinranking.com/coin/nINoAiAdq+amaterasuomikami-omikami/about', 'LP locked via Team Finance; DEXTools audit: honeypot no, tax 0%, renounced'),
 '0x1c4853ec0d55e420002c5efabc7ed8e0ba7a4121': ('N', 'https://coingecko.com/en/coins/kanagawa-nami', ''),
 '0xc5fb36dd2fb59d3b98deff88425a3f425ee469ed': ('N', 'https://coinmarketcap.com/currencies/dejitaru-tsuka', 'listed on CMC; supply lock on Team Finance claimed until 2028'),
 '0xac57de9c1a09fec648e93eb98875b212db0d460b': ('N', 'https://hashex.org/auto-audits/ethereum/0xac57de9c1a09fec648e93eb98875b212db0d460b', 'HashEx auto-audit 0 issues; DEXTools: honeypot no, tax 0%, renounced'),
 '0x298d492e8c1d909d3f63bc4a36c66c64acb3d695': ('N', 'https://hacken.io/audits/polkabridge/', 'audited by Hacken (Jan 2022); little trading now'),
 '0x9e5bd9d9fad182ff0a93ba8085b664bcab00fa68': ('N', 'https://coinmarketcap.com/currencies/dinger-token', ''),
 '0x1045f5ccb01daea4f8eab055f5fcbb7c0e7c89f0': ('N', 'https://www.coingecko.com/en/coins/defiato', 'price about -99.7% from ATH, very low volume'),
 '0xf5581dfefd8fb0e4aec526be659cfab1f8c781da': ('N', 'https://www.coingecko.com/en/coins/hopr', 'developed by the HOPR Association (Swiss non-profit)'),
 '0xa1817b6d8d890f3943b61648992730373b71f156': ('N', 'https://watcher.guru/news/mongoose-coin-it-gained-110000-within-24-hours-after-congressman-mocks-it-another-looming-cryptocurrency-rug-pull', 'only a speculative article ("looming rug pull?"), no confirmed report'),
}
assert set(SCAM) == set(AUD.address), set(AUD.address) ^ set(SCAM)

def f(x):
    try: return float(x)
    except Exception: return None

def verdict(r):
    rep = SCAM[r['address']][0]
    sell_hp = f(r['hp_sell_tax_%']); sell_gp = f(r['gp_sell_tax'])
    sell = max([v for v in [sell_hp, (sell_gp * 100 if sell_gp is not None else None)] if v is not None], default=None)
    if rep == 'Y' or r['hp_is_honeypot'] is True or str(r['gp_is_honeypot']) == '1' or str(r['gp_cannot_sell_all']) == '1' or (sell is not None and sell >= 50):
        return 'likely scam'
    dead = (r['transfers_since_2026_01_01 (max 100)'] or 0) == 0
    risky = (sell is not None and sell >= 10) or str(r['gp_owner_change_balance']) == '1' or str(r['gp_hidden_owner']) == '1'
    if rep == 'Y?' or dead or risky: return 'suspicious'
    return 'legit'

OUTF = AUD.copy()
for i, r in EV.iterrows():
    k = OUTF.address == r['address']
    rep, link, note = SCAM[r['address']]
    OUTF.loc[k, 'still_traded_2026 (Y/N)'] = 'Y' if (r['transfers_since_2026_01_01 (max 100)'] or 0) > 0 else 'N'
    OUTF.loc[k, 'scam_reports_found (Y/N + link)'] = f'{rep} {link}'.strip()
    OUTF.loc[k, 'honeypot_result'] = (f"honeypot={r['hp_is_honeypot']}; buy {r['hp_buy_tax_%']}%; sell {r['hp_sell_tax_%']}%; "
                                      f"liq ${r['hp_liquidity_usd']}; GoPlus sell_tax={r['gp_sell_tax']}" if r['hp_simulation_ok']
                                      else f"simulation failed ({r['hp_error']}); GoPlus honeypot={r['gp_is_honeypot']}, sell_tax={r['gp_sell_tax']}")
    OUTF.loc[k, 'verdict (legit / suspicious / likely scam)'] = verdict(r) + ' (draft)'
    OUTF.loc[k, 'notes'] = note
with pd.ExcelWriter(f'{OUT}/annotation/top20_legit_audit_filled.xlsx') as w:
    OUTF.to_excel(w, sheet_name='audit', index=False)
    EV.to_excel(w, sheet_name='evidence', index=False)
    pd.DataFrame({'rule': [
        'likely scam: credible scam/rug report, OR honeypot.is/GoPlus says honeypot or cannot sell all, OR sell tax >= 50%',
        'suspicious: not likely scam AND (only a low-credibility report, OR no transfer since 2026-01-01, OR sell tax >= 10%, OR owner can change balances / hidden owner)',
        'legit: everything else',
        'Rule fixed before the on-chain evidence (Etherscan, honeypot.is, GoPlus, CoinGecko) was collected; web search done the same day. Verdicts are drafts: the first author checks each one.']}).to_excel(w, sheet_name='verdict_rule', index=False)
print('wrote', f'{OUT}/annotation/top20_legit_audit_filled.xlsx')
print(OUTF[['contract_name', 'still_traded_2026 (Y/N)', 'scam_reports_found (Y/N + link)', 'verdict (legit / suspicious / likely scam)']].to_string(index=False))

In [ ]:
# 11. Export files for the GitHub replication package (paste into a new cell at the END of the notebook, after
#     cells 5f/5g and the annotation cells have run in the same session; no API key needed; runtime < 1 min).
#     Writes to {OUT}/repo_export/  -> download that folder (or zip it, see last lines) and send it.
import os, shutil
EXP = f'{OUT}/repo_export'; os.makedirs(f'{EXP}/annotation_contracts', exist_ok=True)

# (a) grouping used in every cross-validation: one row per contract of the final dataset (2,278)
grp = pd.DataFrame({
    'address': D.address.values, 'label': y, 'year': D.year.values, 'solc_minor': D.solc_minor.values,
    'type_label': D['Type'].values,
    'feature_cluster': G0,                                    # feature-space grouping (706 clusters)
    'hash_L1': [sha(N1[a]) for a in D.address],               # comments and whitespace removed
    'hash_L2': [sha(N2[a]) for a in D.address],               # + literals replaced (exact-duplicate grouping)
    'sim_group_0.9': GS[0.9], 'sim_group_0.8': GS[0.8], 'sim_group_0.7': GS[0.7],  # similarity + feature clusters
})
grp.to_csv(f'{EXP}/groups.csv', index=False)
print('groups.csv', grp.shape, '| feature clusters', grp.feature_cluster.nunique(),
      '| sim 0.9 groups', grp['sim_group_0.9'].nunique())

# (b) verified source of the 61 annotated contracts (C01-C61), named by annotation id
key = pd.read_csv(f'{OUT}/annotation/annotation_key_DO_NOT_OPEN.csv')
key['address'] = key.address.str.lower(); n = 0
for _, r in key.iterrows():
    if r.address in SRCTXT:
        open(f"{EXP}/annotation_contracts/{r['id']}.sol", 'w').write(SRCTXT[r.address]); n += 1
print('annotated sources written:', n, 'of', len(key))

# (c) the annotation files themselves
for f in ['regex_annotation_sheet.xlsx', 'annotation_key_DO_NOT_OPEN.csv']:
    p = f'{OUT}/annotation/{f}'
    if os.path.exists(p): shutil.copy(p, EXP)
if os.path.exists(f'{OUT}/regex_V4_flags.csv'): shutil.copy(f'{OUT}/regex_V4_flags.csv', EXP)
shutil.copy(XLSX, EXP)                                         # rev2_colab_results.xlsx

shutil.make_archive(f'{OUT}/repo_export', 'zip', EXP)
print('ZIP ->', f'{OUT}/repo_export.zip')